# US 4.3 -- DANN Training

This notebook walks through training a Domain-Adversarial Neural Network.
Unlike the baseline (source-only), DANN uses a **gradient reversal layer**
to make the encoder produce domain-invariant features.

## What you will learn

1. How the training loop differs from the baseline
2. The combined loss: segmentation + domain adversarial
3. How to plot lambda schedule and loss curves
4. How to monitor domain confusion during training

## CLI equivalent

```bash
udm-epic4 train --config configs/epic4_dann.yaml
```

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from udm_epic4.models.dann import DANNModel
from udm_epic4.data.multi_domain_dataset import DomainDataset, build_datasets_from_config
from udm_epic4.data.domain_sampler import DomainBatchSampler
from udm_epic4.training.lambda_scheduler import dann_lambda_schedule
from udm_epic4.training.train_dann import train_dann_from_yaml
from udm_epic4.training.train_baseline import bce_dice_loss

---
## 1. DANN Configuration

The DANN config (`configs/epic4_dann.yaml`) extends the baseline config with:

```yaml
model:
  backbone: convnext_tiny
  pretrained: true
  decoder_channels: [256, 128, 64, 32]
  domain_head_hidden: 256          # <-- new: domain classifier hidden dim

training:
  max_epochs: 100
  batch_size: 8                    # 4 source + 4 target per batch
  lambda_max: 1.0                  # <-- new: GRL strength ceiling
  lambda_schedule: dann            # <-- new: sigmoid ramp-up

loss:
  segmentation: bce_dice
  domain: bce                      # <-- new: domain classification loss
  seg_weight: 1.0
  domain_weight: 1.0

data:
  source:
    name: warstein
    images: data/warstein/images
    masks: data/warstein/masks
  targets:                          # <-- new: target domain (unlabeled)
    - name: malaysia
      images: data/malaysia/images
```

---
## 2. How the DANN Training Loop Works

The key differences from the baseline:

### Balanced batches
Each mini-batch contains **equal parts** source and target images, assembled by
`DomainBatchSampler`. This ensures the domain classifier sees both domains in
every step.

### Three loss components

```
total_loss = seg_weight * seg_loss + domain_weight * domain_loss
```

| Loss | Applied to | Purpose |
|------|-----------|--------|
| `seg_loss` | Source samples only | Train segmentation (BCE + Dice) |
| `domain_loss` | All samples (source + target) | Train domain classifier (BCE) |

The GRL ensures that the `domain_loss` gradient is *reversed* before it reaches
the encoder, so the encoder learns to *confuse* the domain classifier.

### Lambda annealing
The GRL strength ramps up over training via `dann_lambda_schedule`.

In [ ]:
# Visualise the lambda schedule over 100 epochs
max_epochs = 100
epochs = np.arange(1, max_epochs + 1)
lambdas = [dann_lambda_schedule((e - 1) / max(max_epochs - 1, 1), lambda_max=1.0)
           for e in epochs]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs, lambdas, linewidth=2, color='#2196F3')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Lambda (GRL strength)', fontsize=12)
ax.set_title('Lambda Schedule over 100 Epochs', fontsize=14)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

---
## 3. Domain Batch Sampler

The `DomainBatchSampler` ensures balanced batches.  It works with a
`ConcatDataset([source, target])` and yields indices such that each batch
has `batch_size // 2` source and `batch_size // 2` target samples.

In [ ]:
# Demonstrate the sampler with synthetic sizes
sampler = DomainBatchSampler(
    source_dataset_size=100,
    target_dataset_size=80,
    batch_size=8,
    drop_last=True,
)
print(f"Sampler: {sampler}")
print(f"Batches per epoch: {len(sampler)}")

# Show first 3 batches
for i, batch_indices in enumerate(sampler):
    if i >= 3:
        break
    src_indices = [idx for idx in batch_indices if idx < 100]
    tgt_indices = [idx - 100 for idx in batch_indices if idx >= 100]
    print(f"  Batch {i}: {len(src_indices)} source + {len(tgt_indices)} target")

---
## 4. Running DANN Training

### Option A: CLI (recommended for full training)

```bash
udm-epic4 train --config configs/epic4_dann.yaml
```

### Option B: Python API

```python
from udm_epic4.training.train_dann import train_dann_from_yaml

best_checkpoint = train_dann_from_yaml("configs/epic4_dann.yaml")
print(f"Best DANN checkpoint: {best_checkpoint}")
```

The training function:
1. Builds source + target datasets from the config.
2. Creates a `DANNModel` with the specified backbone.
3. Uses `DomainBatchSampler` for balanced batches.
4. Trains with mixed precision (if CUDA available).
5. Logs per-epoch: seg_loss, domain_loss, lambda, domain_acc, source_val_f1.

---
## 5. Understanding the Loss Curves

During DANN training, you should monitor three quantities:

In [ ]:
# Simulated training curves (replace with actual logged values)
np.random.seed(42)
epochs_plot = np.arange(1, 101)

# Segmentation loss: decreases over time
seg_loss = 0.8 * np.exp(-0.03 * epochs_plot) + 0.15 + np.random.normal(0, 0.02, 100)

# Domain loss: starts low (easy to classify), increases as GRL kicks in
domain_loss = 0.3 + 0.35 * (1 - np.exp(-0.05 * epochs_plot)) + np.random.normal(0, 0.02, 100)

# Domain accuracy: starts high (~100%), should drop toward 50% (confusion)
domain_acc = 0.95 - 0.40 * (1 - np.exp(-0.04 * epochs_plot)) + np.random.normal(0, 0.03, 100)
domain_acc = np.clip(domain_acc, 0.4, 1.0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Segmentation loss
axes[0].plot(epochs_plot, seg_loss, color='#2196F3', alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Segmentation Loss')
axes[0].grid(True, alpha=0.3)

# Domain loss
axes[1].plot(epochs_plot, domain_loss, color='#FF5722', alpha=0.8)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Domain Loss')
axes[1].grid(True, alpha=0.3)

# Domain accuracy
axes[2].plot(epochs_plot, domain_acc, color='#4CAF50', alpha=0.8)
axes[2].axhline(y=0.5, color='red', linestyle='--', label='Random (ideal)')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Domain Classifier Accuracy')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

fig.suptitle('DANN Training Curves (Simulated)', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

### How to read the curves

| Metric | Healthy training looks like | Problem signs |
|--------|---------------------------|---------------|
| **Seg loss** | Steadily decreases | Diverges or oscillates wildly |
| **Domain loss** | Increases as GRL kicks in | Stays at 0 (GRL not working) |
| **Domain accuracy** | Drops from ~100% toward ~50% | Stays at 100% (no adaptation) |

The **domain accuracy approaching 50%** is the key signal that DANN is working --
it means the encoder has learned features where source and target are indistinguishable.

---
## 6. Quick Forward Pass Demo

In [ ]:
# Create a DANN model and show how the two outputs work
model = DANNModel(backbone="convnext_tiny", pretrained=False)
model.eval()

# Simulate a mixed batch: 4 source + 4 target
batch_images = torch.randn(8, 3, 512, 512)
domain_labels = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1])  # 0=source, 1=target

with torch.no_grad():
    seg_logits, domain_logits = model(batch_images, lambda_val=0.5)

print(f"Segmentation logits: {seg_logits.shape}")
print(f"Domain logits:       {domain_logits.shape}")

# In training, we compute:
# 1. seg_loss = bce_dice_loss(seg_logits[source_mask], masks[source_mask])
# 2. domain_loss = BCE(domain_logits, domain_labels)
domain_bce = nn.BCEWithLogitsLoss()
domain_targets = domain_labels.float().unsqueeze(1)
d_loss = domain_bce(domain_logits, domain_targets)
print(f"Domain loss: {d_loss.item():.4f}")

# Check domain predictions
domain_preds = (torch.sigmoid(domain_logits) > 0.5).long().squeeze(1)
accuracy = (domain_preds == domain_labels).float().mean()
print(f"Domain accuracy: {accuracy.item():.2%}")

---
## Summary

- DANN training uses balanced source/target batches via `DomainBatchSampler`.
- Total loss = segmentation loss (source only) + domain loss (all samples).
- Lambda ramps up from 0 to `lambda_max` via a sigmoid schedule.
- Key metric: domain classifier accuracy dropping toward 50% = domain confusion.
- Use `udm-epic4 train` or `train_dann_from_yaml()` to run training.

**Next:** `epic4_04_domain_analysis.ipynb` -- analyse the learned features with t-SNE.